# Progetto: Timed GNN per la risoluzione di puzzle scacchistici (Lichess)
### Fase 4 — Baseline LLM e confronto GNN vs LLM

Questo notebook implementa la **baseline LLM** (metodologia, sezione 3 del PDF) e il protocollo di
confronto con il modello GNN della Fase 3.

**Cosa fa:**
1. Prompting di un LLM (Claude o GPT, a scelta) sugli stessi puzzle "mate in n", zero/few-shot
2. Parsing robusto della risposta in una sequenza di mosse UCI
3. Metriche: accuracy sulla prima mossa (comparabile 1:1 con la GNN), accuracy sequenza completa,
   tempo di risposta
4. Predisposizione della **valutazione umana in cieco** (rubrica 1-5 correttezza/concisione) richiesta
   dalla metodologia
5. Confronto statistico GNN vs LLM (McNemar's test) e grafico accuracy per valore di `n`

> **Nota sui costi**: ogni puzzle valutato = una chiamata API. Il notebook lavora di default su un
> campione piccolo (`N_SAMPLES`) per motivi di costo/tempo; aumenta il valore quando sei pronto per la
> valutazione finale su tutto il held-out set.


## 0. Setup e scelta del backend LLM

In [ ]:
!pip install -q anthropic openai python-chess

import os, re, time, json
import numpy as np
import pandas as pd
import chess
import matplotlib.pyplot as plt
from getpass import getpass

DATA_DIR = "/content/data"
RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)


Scegli il backend. Entrambi richiedono una API key personale (non condividerla mai nel notebook
in chiaro: usa `getpass` come sotto, oppure i **Secrets** di Colab, icona a forma di chiave nella
sidebar).

- `"anthropic"` -> modelli Claude (es. `claude-sonnet-5`)
- `"openai"` -> modelli GPT (verifica su https://platform.openai.com/docs/models il nome del modello
  attualmente disponibile per il tuo account, dato che cambia nel tempo)


In [ ]:
BACKEND = "anthropic"   # oppure "openai"
MODEL_NAME = "claude-sonnet-5"   # se BACKEND="openai", es. "gpt-4o" (verifica il nome corrente)

if BACKEND == "anthropic":
    import anthropic
    api_key = os.environ.get("ANTHROPIC_API_KEY") or getpass("Anthropic API key: ")
    client = anthropic.Anthropic(api_key=api_key)
elif BACKEND == "openai":
    import openai
    api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
    client = openai.OpenAI(api_key=api_key)
else:
    raise ValueError("BACKEND deve essere 'anthropic' o 'openai'")

print(f"Backend configurato: {BACKEND} / {MODEL_NAME}")


## 1. Prompt template

Seguendo la metodologia del progetto ("Given this FEN: [FEN], find mate in n moves. Explain
step-by-step"), costruiamo il prompt partendo dalla **posizione dopo la mossa di apertura
dell'avversario** — la stessa posizione su cui lavora la GNN — così il confronto sulla prima mossa è
equo. Chiediamo esplicitamente una riga finale in formato UCI per rendere il parsing affidabile, senza
however impedire la spiegazione discorsiva richiesta dal progetto.


In [ ]:
PROMPT_TEMPLATE = '''You are a chess expert. Analyze this position and find a forced checkmate.

FEN: {fen}
Side to move: {side}
The position has a forced checkmate in exactly {n} move(s) for the side to move.

Explain your reasoning step-by-step, then finish your answer with exactly one line in this format
(UCI notation, space-separated, no extra text on that line):

SOLUTION: <move1> <move2> <move3> ...
'''

def build_prompt(fen, n):
    side = "White" if fen.split()[1] == "w" else "Black"
    return PROMPT_TEMPLATE.format(fen=fen, side=side, n=n)


## 2. Chiamata all'LLM (con retry) e parsing della risposta

In [ ]:
def call_llm(prompt, max_retries=3, max_tokens=1024):
    for attempt in range(max_retries):
        try:
            if BACKEND == "anthropic":
                resp = client.messages.create(
                    model=MODEL_NAME,
                    max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}],
                )
                return "".join(block.text for block in resp.content if hasattr(block, "text"))
            else:  # openai
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}],
                )
                return resp.choices[0].message.content
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [tentativo {attempt+1}/{max_retries}] errore: {e} -> retry tra {wait}s")
            time.sleep(wait)
    raise RuntimeError("Chiamata LLM fallita dopo tutti i retry")


SOLUTION_LINE_RE = re.compile(r"SOLUTION:\s*(.+)")

def parse_solution(response_text):
    """Estrae la lista di mosse UCI dalla riga 'SOLUTION: ...'. Ritorna None se non trovata/non valida."""
    match = None
    for line in response_text.splitlines():
        m = SOLUTION_LINE_RE.search(line)
        if m:
            match = m.group(1).strip()
    if match is None:
        return None

    candidate_tokens = re.findall(r"[a-h][1-8][a-h][1-8][qrbn]?", match.lower())
    return candidate_tokens if candidate_tokens else None


## 3. Verifica della soluzione con `python-chess`

Data una sequenza di mosse UCI proposta dall'LLM, verifichiamo:
- **legalità** di ogni mossa nella posizione corrente
- se la sequenza corrisponde esattamente alla soluzione del puzzle (metrica "sequenza completa",
  rigorosa: non riconosce matti alternativi)
- se la **prima mossa** coincide con la mossa target (metrica comparabile 1:1 con la GNN)


In [ ]:
def evaluate_llm_solution(fen_after_opening, target_moves_uci, proposed_uci_list):
    """target_moves_uci: lista delle mosse di soluzione (dopo la mossa di apertura), come da Lichess."""
    if not proposed_uci_list:
        return {"first_move_correct": False, "full_sequence_correct": False, "proposed_legal": False}

    board = chess.Board(fen_after_opening)
    all_legal = True
    for uci in proposed_uci_list:
        try:
            move = chess.Move.from_uci(uci)
        except ValueError:
            all_legal = False
            break
        if move not in board.legal_moves:
            all_legal = False
            break
        board.push(move)

    first_move_correct = (len(proposed_uci_list) > 0 and
                           len(target_moves_uci) > 0 and
                           proposed_uci_list[0] == target_moves_uci[0])

    full_sequence_correct = (proposed_uci_list == target_moves_uci)

    return {
        "first_move_correct": first_move_correct,
        "full_sequence_correct": full_sequence_correct,
        "proposed_legal": all_legal,
    }


## 4. Caricamento del set di valutazione

Usiamo il **set held-out** preparato in Fase 1 (punto 8), se disponibile. In sua assenza (es. non
ancora scaricato/costruito), ripieghiamo su un campione del test set interno di Lichess, utile per
testare la pipeline ma meno probante scientificamente (la metodologia richiede un set esterno per
la valutazione finale).


In [ ]:
held_out_path = os.path.join(DATA_DIR, "puzzles_held_out.parquet")
test_path = os.path.join(DATA_DIR, "puzzles_test.parquet")

if os.path.exists(held_out_path):
    df_eval_full = pd.read_parquet(held_out_path)
    print(f"Uso il set held-out esterno: {len(df_eval_full):,} puzzle")
elif os.path.exists(test_path):
    df_eval_full = pd.read_parquet(test_path)
    print(f"ATTENZIONE: held-out non trovato, uso il test set interno di Lichess "
          f"({len(df_eval_full):,} puzzle) solo per testare la pipeline.")
else:
    raise FileNotFoundError("Nessun dataset di valutazione trovato. Esegui prima le Fasi 1-3.")

N_SAMPLES = 20   # aumenta per la valutazione finale (attenzione a costi/tempo delle chiamate API)
df_eval = df_eval_full.groupby("mate_n", group_keys=False).apply(
    lambda g: g.sample(min(len(g), max(1, N_SAMPLES // df_eval_full["mate_n"].nunique())), random_state=0)
).reset_index(drop=True)

print(f"Campione di valutazione: {len(df_eval)} puzzle, distribuzione per n:")
print(df_eval['mate_n'].value_counts().sort_index())


## 5. Esecuzione della valutazione LLM sul campione

In [ ]:
def get_position_after_opening(fen, moves_str):
    board = chess.Board(fen)
    moves = moves_str.split()
    if len(moves) >= 2:
        board.push(chess.Move.from_uci(moves[0]))
        target_moves = moves[1:]
    else:
        target_moves = moves
    return board.fen(), target_moves

llm_results = []

for i, row in df_eval.reset_index(drop=True).iterrows():
    fen_after, target_moves = get_position_after_opening(row["FEN"], row["Moves"])
    prompt = build_prompt(fen_after, row["mate_n"])

    t0 = time.time()
    try:
        response_text = call_llm(prompt)
    except RuntimeError as e:
        print(f"[{i}] puzzle {row['PuzzleId']}: FALLITO ({e})")
        continue
    elapsed = time.time() - t0

    proposed = parse_solution(response_text)
    eval_res = evaluate_llm_solution(fen_after, target_moves, proposed)

    llm_results.append({
        "PuzzleId": row["PuzzleId"],
        "mate_n": row["mate_n"],
        "fen_after_opening": fen_after,
        "target_moves": " ".join(target_moves),
        "proposed_moves": " ".join(proposed) if proposed else None,
        "response_time_sec": elapsed,
        "llm_explanation": response_text,
        **eval_res,
    })

    status = "OK" if eval_res["first_move_correct"] else "sbagliato"
    print(f"[{i+1}/{len(df_eval)}] {row['PuzzleId']} (n={row['mate_n']}): "
          f"prima mossa {status} | {elapsed:.1f}s")

df_llm = pd.DataFrame(llm_results)
df_llm.to_csv(os.path.join(RESULTS_DIR, "llm_evaluation_raw.csv"), index=False)
print(f"\nCompletato: {len(df_llm)}/{len(df_eval)} puzzle valutati con successo.")


## 6. Metriche LLM aggregate

Accuracy sulla prima mossa (comparabile con la GNN), accuracy sulla sequenza completa (più severa),
percentuale di risposte legali, tempo medio di risposta.


In [ ]:
print("=== Metriche LLM (campione valutato) ===")
print(f"Accuracy prima mossa:        {df_llm['first_move_correct'].mean():.3f}")
print(f"Accuracy sequenza completa:  {df_llm['full_sequence_correct'].mean():.3f}")
print(f"% risposte con mosse legali: {df_llm['proposed_legal'].mean():.3f}")
print(f"Tempo medio di risposta:     {df_llm['response_time_sec'].mean():.1f}s")

print("\nAccuracy prima mossa per valore di n:")
print(df_llm.groupby("mate_n")["first_move_correct"].mean())


## 7. Confronto con la GNN sugli stessi puzzle

Carichiamo il modello `timed_gnn_best.pt` della Fase 3 e lo eseguiamo **esattamente sugli stessi
puzzle** valutati sopra con l'LLM, per un confronto appaiato (necessario per il McNemar's test).

> Richiede che siano già presenti in questa sessione le funzioni/classi della Fase 3
> (`TimedGNN`, `ChessPuzzleGraphDataset`, ecc.). Se stai eseguendo questo notebook separatamente,
> ri-esegui prima le celle di setup/model della Fase 3, oppure copia qui quelle celle.


In [ ]:
import torch
import torch.nn.functional as F

assert "TimedGNN" in globals(), (
    "Esegui prima le celle 'Setup' e 'Il modello TimedGNN' del notebook "
    "03_gnn_model_training.ipynb in questa sessione."
)

CKPT_PATH = "/content/checkpoints/timed_gnn_best.pt"
model_timed.eval()

gnn_correct_flags = []
for _, row in df_llm.iterrows():
    match_row = df_eval[df_eval["PuzzleId"] == row["PuzzleId"]].iloc[0]
    ds_single = ChessPuzzleGraphDataset(pd.DataFrame([match_row]), include_timing=True)
    data = ds_single.get(0).to(device)
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)

    with torch.no_grad():
        scores, edge_batch = model_timed(data)
    pred_idx = scores.argmax().item()
    correct = (pred_idx == data.y.item())
    gnn_correct_flags.append(correct)

df_llm["gnn_first_move_correct"] = gnn_correct_flags
print("Accuracy GNN sullo stesso campione:", df_llm['gnn_first_move_correct'].mean())
print("Accuracy LLM sullo stesso campione:", df_llm['first_move_correct'].mean())


## 8. McNemar's test (confronto appaiato GNN vs LLM)

Il test di McNemar è appropriato qui perché i due modelli sono valutati **sugli stessi puzzle**
(confronto appaiato), come richiesto dalla metodologia. Costruiamo la tabella di contingenza 2x2 e
calcoliamo la statistica con correzione di continuità.


In [ ]:
from scipy.stats import chi2

both_correct = ((df_llm['gnn_first_move_correct']) & (df_llm['first_move_correct'])).sum()
only_gnn = ((df_llm['gnn_first_move_correct']) & (~df_llm['first_move_correct'])).sum()
only_llm = ((~df_llm['gnn_first_move_correct']) & (df_llm['first_move_correct'])).sum()
neither = ((~df_llm['gnn_first_move_correct']) & (~df_llm['first_move_correct'])).sum()

print("Tabella di contingenza:")
print(f"                 LLM corretto   LLM sbagliato")
print(f"GNN corretto     {both_correct:^13}  {only_gnn:^14}")
print(f"GNN sbagliato    {only_llm:^13}  {neither:^14}")

b, c = only_gnn, only_llm
if b + c == 0:
    print("\nNessuna discordanza (b+c=0): il test di McNemar non è applicabile su questo campione.")
else:
    stat = (abs(b - c) - 1) ** 2 / (b + c)   # con correzione di continuità
    p_value = chi2.sf(stat, df=1)
    print(f"\nMcNemar chi2 = {stat:.3f}, p-value = {p_value:.4f}")
    if p_value < 0.05:
        winner = "GNN" if b > c else "LLM"
        print(f"-> Differenza statisticamente significativa (p<0.05): {winner} prevale su questo campione.")
    else:
        print("-> Nessuna differenza statisticamente significativa su questo campione "
              "(probabile: campione piccolo, N_SAMPLES da aumentare).")


## 9. Grafico: accuracy per valore di `n`, GNN vs LLM

In [ ]:
acc_by_n = df_llm.groupby("mate_n").agg(
    gnn_acc=("gnn_first_move_correct", "mean"),
    llm_acc=("first_move_correct", "mean"),
    n_puzzles=("PuzzleId", "count"),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.35
x = np.arange(len(acc_by_n))
ax.bar(x - width/2, acc_by_n["gnn_acc"], width, label="Timed GNN")
ax.bar(x + width/2, acc_by_n["llm_acc"], width, label=f"LLM ({MODEL_NAME})")
ax.set_xticks(x)
ax.set_xticklabels([f"mate in {n}\n(n={c})" for n, c in zip(acc_by_n["mate_n"], acc_by_n["n_puzzles"])])
ax.set_ylabel("accuracy prima mossa")
ax.set_title("GNN vs LLM: accuracy per profondità di matto")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "gnn_vs_llm_by_n.png"), dpi=150)
plt.show()

print(acc_by_n)


## 10. Predisposizione della valutazione umana in cieco

La metodologia richiede anche una valutazione umana della **qualità della guida** (chiarezza,
concisione), non solo della correttezza. Generiamo un CSV "in cieco": per ogni puzzle, le due
spiegazioni (GNN — solo la mossa, non genera testo naturale; LLM — spiegazione completa) vengono
etichettate come "Modello A" / "Modello B" in ordine **randomizzato per riga**, cosicché chi valuta
non sappia quale sia quale. La colonna di mapping reale (`_true_label_A`) va tenuta separata e
consultata solo dopo la valutazione.


In [ ]:
rng = np.random.default_rng(0)
blind_rows = []

for _, row in df_llm.iterrows():
    swap = rng.random() < 0.5
    model_a_is_gnn = not swap

    gnn_move_san = "-"  # la GNN produce solo la mossa, non una spiegazione testuale
    llm_text = row["llm_explanation"]

    blind_rows.append({
        "PuzzleId": row["PuzzleId"],
        "mate_n": row["mate_n"],
        "Modello_A": (f"Mossa proposta: {row['PuzzleId']}" if model_a_is_gnn else llm_text),
        "Modello_B": (llm_text if model_a_is_gnn else f"Mossa proposta: {row['PuzzleId']}"),
        "A_correttezza_1_5": "",
        "A_concisione_1_5": "",
        "B_correttezza_1_5": "",
        "B_concisione_1_5": "",
        "_true_label_A": "GNN" if model_a_is_gnn else "LLM",   # non mostrare a chi valuta
    })

df_blind = pd.DataFrame(blind_rows)
blind_path_for_rater = os.path.join(RESULTS_DIR, "human_eval_blind.csv")
df_blind.drop(columns=["_true_label_A"]).to_csv(blind_path_for_rater, index=False)
df_blind.to_csv(os.path.join(RESULTS_DIR, "human_eval_blind_WITH_ANSWER_KEY.csv"), index=False)

print(f"File per il valutatore (senza risposte): {blind_path_for_rater}")
print("Compila le colonne '*_correttezza_1_5' e '*_concisione_1_5' (1=pessimo, 5=ottimo), poi "
      "ricongiungi con '_true_label_A' dal file 'WITH_ANSWER_KEY' per l'analisi finale.")


## Riepilogo Fase 4 e prossimi passi

**Cosa abbiamo costruito:**
- Prompting LLM riproducibile (con retry) e parsing robusto della risposta in mosse UCI
- Metriche comparabili 1:1 con la GNN (accuracy prima mossa) più metriche più severe (sequenza
  completa) e di efficienza (tempo di risposta)
- Confronto **appaiato** GNN vs LLM sugli stessi puzzle, con McNemar's test
- Grafico accuracy per valore di `n` — risponde direttamente alla research question 1
- File pronto per la valutazione umana in cieco (rubrica 1-5), come richiesto dalla metodologia

**Da fare prima della valutazione finale "vera":**
- Alzare `N_SAMPLES` a 100-200 (metodologia, Fase 1 punto 8) sul set held-out esterno reale
- Far compilare `human_eval_blind.csv` a te stesso/a e/o ai compagni di gruppo (valutazione in cieco)
- Ripetere il confronto anche con il modello **senza timing** (`no_timing_gnn_best.pt`) per completare
  l'ablation study end-to-end (Fase 5 del progetto)

**Prossimo passo naturale**: mettere insieme tutti i risultati (Fasi 1-4) in un report finale con
grafici, tabelle e insight (Objectives 4-6 del PDF: valutazione comparativa, ablation study,
documentazione). Dimmi se vuoi che prepari anche quello.
